In [10]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch

ROOT              = Path(".").resolve()
OHLCV_DIR         = ROOT / "data/OHLCV"
MACRO_SIGNALS_DIR = ROOT / "data/macro_signals"
TENSORS_DIR       = ROOT / "data/tensors"
PREPROCESSING_DIR = ROOT / "data/preprocessing"
MODEL_INPUTS_DIR  = ROOT / "data/model_inputs"

LGBM_PRED_PATH             = MODEL_INPUTS_DIR / "lgbm_predictions.csv"
DAILY_CLASSIFICATIONS_PATH = MODEL_INPUTS_DIR / "daily_risk_classifications.csv"

TENSORS_DIR.mkdir(parents=True, exist_ok=True)
PREPROCESSING_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_NAMES = [
    "ret_1d",
    "ret_5d",
    "ret_20d",
    "lgbm_pred",
    "lgbm_pred_valid",    # new: missingness indicator
    "gk_vol_20d",
    "skew_20d",
    "kurt_20d",
    "rel_volume",
    "avg_pairwise_corr",
]

TRAIN_END = "2021-12-31"
VAL_END   = "2022-12-31"
RETURN_CLAMP      = 0.25
VOLUME_AVG_WINDOW = 20
CORR_WINDOW       = 20
MA_SHORT          = 20
MA_LONG           = 60
CONIA_DELTA_WINDOW = 60
RISK_PROFILE      = "balanced"

print(f"ROOT         : {ROOT}")
print(f"FEATURE_NAMES: {FEATURE_NAMES}")
print(f"F            : {len(FEATURE_NAMES)}")

ROOT         : C:\Users\mirae\Desktop\Personalization_Engine
FEATURE_NAMES: ['ret_1d', 'ret_5d', 'ret_20d', 'lgbm_pred', 'lgbm_pred_valid', 'gk_vol_20d', 'skew_20d', 'kurt_20d', 'rel_volume', 'avg_pairwise_corr']
F            : 10


In [11]:
# Load pre-built panels from macro_signals/ — no need to reprocess OHLCV
prices_df = pd.read_csv(MACRO_SIGNALS_DIR / "all_close_prices.csv",
                        index_col=0, parse_dates=True)
open_df   = pd.read_csv(MACRO_SIGNALS_DIR / "all_open_prices.csv",
                        index_col=0, parse_dates=True)
high_df   = pd.read_csv(MACRO_SIGNALS_DIR / "all_high_prices.csv",
                        index_col=0, parse_dates=True)
low_df    = pd.read_csv(MACRO_SIGNALS_DIR / "all_low_prices.csv",
                        index_col=0, parse_dates=True)
volume_df = pd.read_csv(MACRO_SIGNALS_DIR / "all_volume.csv",
                        index_col=0, parse_dates=True)
validity_mask_final = pd.read_csv(MACRO_SIGNALS_DIR / "validity_mask.csv",
                                   index_col=0, parse_dates=True)

tickers = sorted(prices_df.columns.tolist())
print(f"Panels loaded: {prices_df.shape}  tickers: {len(tickers)}")
print(f"Date range   : {prices_df.index.min().date()} → {prices_df.index.max().date()}")

Panels loaded: (2767, 31)  tickers: 31
Date range   : 2014-11-11 → 2026-03-31


In [12]:
with open(PREPROCESSING_DIR / "norm_stats.json") as f:
    norm_stats_existing = json.load(f)

with open(PREPROCESSING_DIR / "market_signal_stats.json") as f:
    market_signal_stats = json.load(f)

with open(PREPROCESSING_DIR / "macro_signal_stats.json") as f:
    macro_signal_stats = json.load(f)

_splits       = np.load(PREPROCESSING_DIR / "data_splits.npz")
date_index    = _splits["dates"].tolist()
_split_ranges = {k: _splits[k].tolist() for k in ("train", "val", "test")}

# Reconstruct date series from the existing trimmed dataset
prices_trimmed = prices_df.loc[date_index[0]:date_index[-1]]
tickers        = sorted(prices_trimmed.columns.tolist())

train_dates = pd.DatetimeIndex([d for d in prices_trimmed.index
                                 if d <= pd.Timestamp(TRAIN_END)])
val_dates   = pd.DatetimeIndex([d for d in prices_trimmed.index
                                 if pd.Timestamp(TRAIN_END) < d <= pd.Timestamp(VAL_END)])
test_dates  = pd.DatetimeIndex([d for d in prices_trimmed.index
                                 if d > pd.Timestamp(VAL_END)])

print(f"Trimmed panel : {prices_trimmed.shape}")
print(f"Train dates   : {len(train_dates)}")
print(f"Val dates     : {len(val_dates)}")
print(f"Test dates    : {len(test_dates)}")
print(f"Existing norm_stats features: {list(norm_stats_existing.keys())}")

Trimmed panel : (2607, 31)
Train dates   : 1579
Val dates     : 244
Test dates    : 784
Existing norm_stats features: ['ret_1d', 'ret_5d', 'ret_20d', 'lgbm_pred', 'gk_vol_20d', 'skew_20d', 'kurt_20d', 'rel_volume', 'avg_pairwise_corr']


In [13]:
# Load LightGBM predictions with missingness indicator
if LGBM_PRED_PATH.exists():
    lgbm_raw = pd.read_csv(LGBM_PRED_PATH, index_col=0, parse_dates=True)
    lgbm_raw.index = pd.to_datetime(lgbm_raw.index).normalize()
    lgbm_raw = lgbm_raw[~lgbm_raw.index.duplicated(keep="last")].sort_index()
    lgbm_aligned = lgbm_raw.reindex(index=prices_df.index, columns=tickers)

    # Missingness indicator: 1 = real prediction, 0 = missing
    lgbm_valid   = lgbm_aligned.notna().astype(float)
    lgbm_filled  = lgbm_aligned.fillna(0.0)

    n_valid = lgbm_aligned.notna().sum().sum()
    n_total = lgbm_aligned.size
    print(f"✓ LGBM loaded: {n_valid:,} valid cells ({n_valid/n_total*100:.1f}% coverage)")

    # Check which OHLCV ticker is missing from LGBM
    missing = set(tickers) - set(lgbm_raw.columns)
    print(f"  Tickers missing from LGBM: {missing}")
    print(f"  Avg valid stocks per day : {lgbm_aligned.notna().sum(axis=1).mean():.1f}")
else:
    raise FileNotFoundError(f"LGBM predictions not found: {LGBM_PRED_PATH}")

# Trim to exact date_index used by the tensor (not prices_df range)
# Trim to exact date_index used by the tensor (not prices_df range)
date_index_ts      = pd.DatetimeIndex(date_index)
lgbm_pred_trimmed  = lgbm_filled.reindex(index=date_index_ts, columns=tickers).fillna(0.0)
lgbm_valid_trimmed = lgbm_valid.reindex(index=date_index_ts, columns=tickers).fillna(0.0)

print(f"\nTrimmed lgbm_pred  : {lgbm_pred_trimmed.shape}")
print(f"Trimmed lgbm_valid : {lgbm_valid_trimmed.shape}")
assert lgbm_pred_trimmed.shape[0] == len(date_index), \
    f"Row mismatch: lgbm has {lgbm_pred_trimmed.shape[0]} rows, expected {len(date_index)}"

✓ LGBM loaded: 75,630 valid cells (88.2% coverage)
  Tickers missing from LGBM: {'FWRY'}
  Avg valid stocks per day : 27.3

Trimmed lgbm_pred  : (2647, 31)
Trimmed lgbm_valid : (2647, 31)


In [14]:
# Load existing feature tensor built by previous pipeline
existing_ft = torch.load(TENSORS_DIR / "feature_tensor.pt", weights_only=False).numpy()

OLD_FEATURE_NAMES = [
    "ret_1d", "ret_5d", "ret_20d", "lgbm_pred",
    "gk_vol_20d", "skew_20d", "kurt_20d", "rel_volume", "avg_pairwise_corr",
]

T = existing_ft.shape[0]
N = existing_ft.shape[1]
F_new = len(FEATURE_NAMES)   # 10

print(f"Existing tensor : {existing_ft.shape}  ({len(OLD_FEATURE_NAMES)} features)")
print(f"New tensor will : ({T}, {N}, {F_new})  ({F_new} features)")

# Build new tensor by inserting lgbm_pred_valid after lgbm_pred
new_ft = np.zeros((T, N, F_new), dtype=np.float32)

old_lgbm_idx   = OLD_FEATURE_NAMES.index("lgbm_pred")
new_lgbm_idx   = FEATURE_NAMES.index("lgbm_pred")
new_valid_idx  = FEATURE_NAMES.index("lgbm_pred_valid")

# Copy all features that existed before lgbm_pred
for f_idx, name in enumerate(OLD_FEATURE_NAMES):
    if name == "lgbm_pred":
        continue
    new_f_idx = FEATURE_NAMES.index(name)
    new_ft[:, :, new_f_idx] = existing_ft[:, :, f_idx]

# Compute normalized lgbm_pred using existing norm_stats
lgbm_mean = norm_stats_existing["lgbm_pred"]["mean"]
lgbm_std  = norm_stats_existing["lgbm_pred"]["std"]

lgbm_norm = (lgbm_pred_trimmed - lgbm_mean) / lgbm_std
lgbm_norm_values = lgbm_norm[tickers].values.astype(np.float32)
lgbm_norm_values = np.nan_to_num(lgbm_norm_values, nan=0.0)

new_ft[:, :, new_lgbm_idx]  = lgbm_norm_values
new_ft[:, :, new_valid_idx] = lgbm_valid_trimmed[tickers].values.astype(np.float32)

# Apply existing mask: zero features for invalid stocks
mask_tensor = torch.load(TENSORS_DIR / "mask_tensor.pt", weights_only=False).numpy()
for f_idx in range(F_new):
    new_ft[:, :, f_idx] *= mask_tensor

print(f"\nNew feature tensor : {new_ft.shape}")
print(f"  lgbm_pred  channel stats: mean={new_ft[:,:,new_lgbm_idx].mean():.4f}  "
      f"std={new_ft[:,:,new_lgbm_idx].std():.4f}")
print(f"  lgbm_valid channel stats: mean={new_ft[:,:,new_valid_idx].mean():.4f}  "
      f"(fraction with valid predictions)")

Existing tensor : (2647, 31, 9)  (9 features)
New tensor will : (2647, 31, 10)  (10 features)

New feature tensor : (2647, 31, 10)
  lgbm_pred  channel stats: mean=-0.4429  std=2.3553
  lgbm_valid channel stats: mean=0.8786  (fraction with valid predictions)


In [15]:
# Build updated norm_stats including lgbm_pred_valid
norm_stats_new = dict(norm_stats_existing)   # copy all existing stats
norm_stats_new["lgbm_pred_valid"] = {"mean": 0.0, "std": 1.0}  # binary, no normalization

# Reorder to match FEATURE_NAMES order
norm_stats_ordered = {name: norm_stats_new[name] for name in FEATURE_NAMES
                      if name in norm_stats_new}

# Save updated norm_stats
with open(PREPROCESSING_DIR / "norm_stats.json", "w") as f:
    json.dump(norm_stats_ordered, f, indent=2)
print(f"✓ norm_stats.json updated with {len(norm_stats_ordered)} features")

# Save new feature tensor
new_ft_pt = torch.from_numpy(new_ft)
torch.save(new_ft_pt, TENSORS_DIR / "feature_tensor.pt")
print(f"✓ feature_tensor.pt saved: {new_ft_pt.shape}")

# Verify
check = torch.load(TENSORS_DIR / "feature_tensor.pt", weights_only=False)
assert check.shape == (T, N, F_new),    f"Shape mismatch: {check.shape}"
assert torch.isfinite(check).all(),      "Contains inf/nan"
assert check.shape[2] == len(FEATURE_NAMES), "Feature count mismatch"

print(f"\n✅ Tensor rebuild complete.")
print(f"   Shape     : {check.shape}")
print(f"   Features  : {FEATURE_NAMES}")
print(f"   lgbm_pred coverage in test set:")

test_start, test_end = _split_ranges["test"]
test_valid = new_ft[test_start:test_end, :, new_valid_idx]
print(f"     Avg stocks with valid pred/day: {test_valid.sum(axis=1).mean():.1f} / {N}")
print(f"     Days with all {N} stocks valid: {(test_valid.sum(axis=1) == N).sum()}")

✓ norm_stats.json updated with 10 features
✓ feature_tensor.pt saved: torch.Size([2647, 31, 10])

✅ Tensor rebuild complete.
   Shape     : torch.Size([2647, 31, 10])
   Features  : ['ret_1d', 'ret_5d', 'ret_20d', 'lgbm_pred', 'lgbm_pred_valid', 'gk_vol_20d', 'skew_20d', 'kurt_20d', 'rel_volume', 'avg_pairwise_corr']
   lgbm_pred coverage in test set:
     Avg stocks with valid pred/day: 28.5 / 31
     Days with all 31 stocks valid: 0


In [18]:
def portfolio_diagnostics(allocations_df, tickers, version, fold_name):
    """
    Comprehensive portfolio diagnostics on allocation CSV.
    Reports ENH, HHI, concentration, entropy, active share, stability.
    """
    import numpy as np
    import pandas as pd
    from scipy.stats import entropy as scipy_entropy

    # Drop metadata columns — remaining columns are weights
    weight_cols = [c for c in allocations_df.columns
                   if c not in ["episode", "rebalance", "date"]]
    W = allocations_df[weight_cols].values   # [T, N+1] including cash

    # Threshold analysis: how much of the portfolio is dust?
    DUST_THRESHOLD = 1e-4   # 1 basis point

    print(f"\n{'='*65}")
    print(f"  PORTFOLIO DIAGNOSTICS — {version} {fold_name}")
    print(f"{'='*65}")

    # 1. Dust analysis
    n_dust     = (W < DUST_THRESHOLD).sum(axis=1).mean()
    pct_dust   = (W[W < DUST_THRESHOLD].sum() / W.sum()) * 100
    print(f"\n-- Dust Analysis (positions < {DUST_THRESHOLD:.0e}) --")
    print(f"  Avg positions below threshold : {n_dust:.1f} / {W.shape[1]}")
    print(f"  % portfolio weight in dust    : {pct_dust:.4f}%")
    print(f"  Min weight observed           : {W[W > 0].min():.2e}")
    print(f"  Max weight observed           : {W.max():.4f}")

    # 2. ENH and HHI
    hhi  = (W ** 2).sum(axis=1)
    enh  = 1 / hhi
    print(f"\n-- Concentration (ENH / HHI) --")
    print(f"  ENH   mean : {enh.mean():.2f}  |  min : {enh.min():.2f}  |  max : {enh.max():.2f}")
    print(f"  HHI   mean : {hhi.mean():.4f}  |  max : {hhi.max():.4f}")
    print(f"  Reference  : ENH={W.shape[1]:.0f} (uniform), "
          f"ENH={1/(W.shape[1]*(0.20**2)):.1f} (all at 20% cap)")

    # 3. Top-N concentration
    for n in [1, 3, 5]:
        top_n = np.sort(W, axis=1)[:, -n:].sum(axis=1)
        print(f"  Top-{n} weight : mean={top_n.mean()*100:.1f}%  "
              f"max={top_n.max()*100:.1f}%  min={top_n.min()*100:.1f}%")

    # 4. Weight entropy
    # Clip dust to avoid log(0); normalize to ensure sum=1 per row
    W_safe  = np.where(W < 1e-10, 1e-10, W)
    W_norm  = W_safe / W_safe.sum(axis=1, keepdims=True)
    ent     = np.array([scipy_entropy(row) for row in W_norm])
    max_ent = np.log(W.shape[1])   # entropy of uniform distribution
    rel_ent = ent / max_ent        # 1.0 = maximally diversified

    print(f"\n-- Weight Entropy --")
    print(f"  Mean entropy        : {ent.mean():.3f}  (max possible: {max_ent:.3f})")
    print(f"  Relative entropy    : {rel_ent.mean():.3f}  (1.0 = uniform, 0.0 = single stock)")
    print(f"  Min relative entropy: {rel_ent.min():.3f}")

    # 5. Active share vs equal weight
    stock_cols = [c for c in weight_cols if c != "cash"]
    W_stocks   = allocations_df[stock_cols].values
    n_stocks   = W_stocks.shape[1]

    active_shares = []
    for row in W_stocks:
        mask    = row > DUST_THRESHOLD   # only count meaningful positions
        n_valid = mask.sum()
        if n_valid > 0:
            eq_w = np.where(mask, 1.0 / n_valid, 0.0)
        else:
            eq_w = np.ones(n_stocks) / n_stocks
        active_shares.append(0.5 * np.abs(row - eq_w).sum())

    active_shares = np.array(active_shares)
    print(f"\n-- Active Share vs Profile-Filtered EQW --")
    print(f"  Mean active share   : {active_shares.mean()*100:.1f}%")
    print(f"  Max active share    : {active_shares.max()*100:.1f}%")
    print(f"  Reference           : 0% = pure EQW, 100% = no overlap")

    # 6. Turnover distribution
    turnovers = []
    sorted_df = allocations_df.sort_values(["episode", "rebalance"])
    for ep, grp in sorted_df.groupby("episode"):
        W_ep = grp[weight_cols].values
        for i in range(1, len(W_ep)):
            turnovers.append(np.abs(W_ep[i] - W_ep[i-1]).sum())
    turnovers = np.array(turnovers)
    print(f"\n-- Turnover Distribution (full round-trip) --")
    print(f"  Mean   : {turnovers.mean()*100:.1f}%  |  Std: {turnovers.std()*100:.1f}%")
    print(f"  Median : {np.median(turnovers)*100:.1f}%")
    print(f"  % steps at cap (>79%) : {(turnovers > 0.79).mean()*100:.1f}%")
    print(f"  % steps low (<40%)    : {(turnovers < 0.40).mean()*100:.1f}%")

    # 7. Logit explosion proxy: ratio of max to 10th percentile weight
    # High ratio = logits are diverging
    w_max  = np.sort(W, axis=1)[:, -1]
    w_p10 = np.array([
    np.percentile(row[row > 1e-10], 10) if (row > 1e-10).any() else 1e-10
    for row in W
    ])
    ratio = w_max / np.where(w_p10 > 0, w_p10, 1e-10)
    print(f"\n-- Logit Explosion Proxy (max_weight / 10th_pct_weight) --")
    print(f"  Mean ratio  : {ratio.mean():.1f}x")
    print(f"  Max ratio   : {ratio.max():.1f}x")
    print(f"  Reference   : ratio < 10x healthy, > 100x concerning, > 1000x severe")

    # 8. Allocation stability (weight autocorrelation)
    stability_scores = []
    for col in stock_cols:
        w_series = allocations_df[col].values
        if w_series.std() > 1e-6:
            corr = np.corrcoef(w_series[:-1], w_series[1:])[0, 1]
            stability_scores.append(corr)
    print(f"\n-- Allocation Stability (lag-1 autocorrelation of weights) --")
    if stability_scores:
        print(f"  Mean autocorr : {np.mean(stability_scores):.3f}  "
              f"(1.0 = never changes, 0.0 = random)")
        print(f"  Min autocorr  : {np.min(stability_scores):.3f}")

    # 9. Meaningful position count
    meaningful = (W_stocks > DUST_THRESHOLD).sum(axis=1)
    print(f"\n-- Meaningful Position Count (weight > {DUST_THRESHOLD:.0e}) --")
    print(f"  Mean : {meaningful.mean():.1f}  |  Min : {meaningful.min()}  "
          f"|  Max : {meaningful.max()}")
    print(f"  % rebalances with < 5 meaningful positions  : "
          f"{(meaningful < 5).mean()*100:.1f}%")
    print(f"  % rebalances with > 15 meaningful positions : "
          f"{(meaningful > 15).mean()*100:.1f}%")

    return {
        "enh_mean": float(enh.mean()), "hhi_mean": float(hhi.mean()),
        "rel_entropy_mean": float(rel_ent.mean()),
        "active_share_mean": float(active_shares.mean()),
        "pct_at_turnover_cap": float((turnovers > 0.79).mean()),
        "logit_ratio_mean": float(ratio.mean()),
        "meaningful_positions_mean": float(meaningful.mean()),
    }

from pathlib import Path
alloc_path = Path("model_versions") / "v3_soft_masking" / "fold_4" / "portfolio_allocations.csv"
allocations_df = pd.read_csv(alloc_path)
diag = portfolio_diagnostics(allocations_df, tickers, "v3_soft_masking", "fold_4")


  PORTFOLIO DIAGNOSTICS — v3_soft_masking fold_4

-- Dust Analysis (positions < 1e-04) --
  Avg positions below threshold : 6.4 / 32
  % portfolio weight in dust    : 0.0068%
  Min weight observed           : 8.11e-16
  Max weight observed           : 0.2217

-- Concentration (ENH / HHI) --
  ENH   mean : 11.15  |  min : 6.42  |  max : 18.80
  HHI   mean : 0.0929  |  max : 0.1559
  Reference  : ENH=32 (uniform), ENH=0.8 (all at 20% cap)
  Top-1 weight : mean=15.8%  max=22.2%  min=8.9%
  Top-3 weight : mean=41.4%  max=60.0%  min=24.9%
  Top-5 weight : mean=59.2%  max=84.5%  min=37.3%

-- Weight Entropy --
  Mean entropy        : 2.628  (max possible: 3.466)
  Relative entropy    : 0.758  (1.0 = uniform, 0.0 = single stock)
  Min relative entropy: 0.584

-- Active Share vs Profile-Filtered EQW --
  Mean active share   : 44.7%
  Max active share    : 65.6%
  Reference           : 0% = pure EQW, 100% = no overlap

-- Turnover Distribution (full round-trip) --
  Mean   : 79.9%  |  Std: 1.1